In [51]:
import itertools
import numpy as np
import pandas as pd



pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [52]:
cd ..

/Users/manilesrun/Desktop/Queryanalyser/QA/results_off_topics/dummy_general


In [53]:
ls

bi-encoder_BAAI-bge-large-en-v1.5.csv
bi-encoder_sentence-transformers-all-MiniLM-L6-v2.csv
cleaned_data_script.py
colbert_lightonai-Reason-ModernColBERT.csv
colbert_LiquidAI-LFM2-ColBERT-350M.csv
cross-encoder_mixedbread-ai-mxbai-rerank-large-v2.csv
cross-encoder_Qwen-Qwen3-Reranker-4B.csv
unpacked/


In [60]:
cd unpacked

/Users/manilesrun/Desktop/Queryanalyser/QA/results_off_topics/dummy_general/unpacked


In [61]:
ls

bi-encoder_BAAI-bge-large-en-v1.5.csv
bi-encoder_sentence-transformers-all-MiniLM-L6-v2.csv
chopped/
colbert_lightonai-Reason-ModernColBERT.csv
colbert_LiquidAI-LFM2-ColBERT-350M.csv
cross-encoder_mixedbread-ai-mxbai-rerank-large-v2.csv
cross-encoder_Qwen-Qwen3-Reranker-4B.csv
Untitled.ipynb


In [70]:
model_name = "cross-encoder_mixedbread-ai-mxbai-rerank-large-v2.csv"
df = pd.read_csv(model_name)

In [71]:
df.head()

,routing_method,query,abstract_algebra_scores,anatomy_scores,astronomy_scores,business_ethics_scores,college_computer_science_scores,high_school_chemistry_scores,high_school_geography_scores,jurisprudence_scores
0,cross-encoder,i'd like to make an appointment to get my oil changed,0.008062,0.002632,0.005555,0.007577,0.005555,0.002981,0.004332,0.012432
1,cross-encoder,i want you to know it's a true statement,0.191933,0.075858,0.173288,0.407333,0.212069,0.071591,0.132964,0.637031
2,cross-encoder,how much gas does it take to get to jackson,0.027585,0.019124,0.039639,0.010328,0.020332,0.132964,0.140336,0.090093
3,cross-encoder,how late is it now in ourense,0.042088,0.016915,0.039639,0.014957,0.025957,0.022977,0.071591,0.042088
4,cross-encoder,i would like you to suggest a chinese dish,0.012432,0.004070,0.004070,0.007577,0.006290,0.004070,0.009708,0.004905


In [64]:
scores = df.filter(like="_scores").values                       
scores.shape

(100, 8)

In [65]:
# import math 

# math.comb(8, 4)

In [66]:
names = [c.replace("_scores", "") for c in df.filter(like="_scores").columns]
k = 4

rows = []


# (0,1,2) (0,1,3)...(5,6,7)
# i = 1, cols = (0, 1, 2)

for i, cols in enumerate(itertools.combinations(range(len(names)), k), start=1):
    S = scores[:, cols]
    gap = S.max(axis=1) - np.median(S, axis=1)
    rows.append({
        "group": i,
        "specialists": ", ".join(names[j] for j in cols),
        "gap_cutoff": np.percentile(gap, 90),
    })

groups = pd.DataFrame(rows)
groups

,group,specialists,gap_cutoff
0,1,"abstract_algebra, anatomy, astronomy, business_ethics",0.225367
1,2,"abstract_algebra, anatomy, astronomy, college_computer_science",0.168552
2,3,"abstract_algebra, anatomy, astronomy, high_school_chemistry",0.130425
3,4,"abstract_algebra, anatomy, astronomy, high_school_geography",0.157005
4,5,"abstract_algebra, anatomy, astronomy, jurisprudence",0.204699
5,6,"abstract_algebra, anatomy, business_ethics, college_computer_science",0.207184
6,7,"abstract_algebra, anatomy, business_ethics, high_school_chemistry",0.251547
7,8,"abstract_algebra, anatomy, business_ethics, high_school_geography",0.245308
8,9,"abstract_algebra, anatomy, business_ethics, jurisprudence",0.221216
9,10,"abstract_algebra, anatomy, college_computer_science, high_school_chemistry",0.184506


In [48]:
print("\naverage of all groups:", round(groups["gap_cutoff"].mean(), 3))


average of all groups: 0.001


In [49]:
def average_cutoffs(scores, ks=range(3, 9), percentile=90):
    """Average gap cutoff for each k, over every group of k specialists."""
    result = {}
    for k in ks:
        cutoffs = []
        for cols in itertools.combinations(range(scores.shape[1]), k):
            S = scores[:, cols]
            gap = S.max(axis=1) - np.median(S, axis=1)
            cutoffs.append(np.percentile(gap, percentile))
        result[k] = np.mean(cutoffs)

    return pd.DataFrame([result], index=["Average cutoff"]).rename_axis("k", axis=1)

In [50]:
df = pd.read_csv(model_name)
scores = df.filter(like="_scores").values

average_cutoffs(scores)

k,3,4,5,6,7,8
Average cutoff,0.000984,0.001112,0.001261,0.001332,0.001454,0.00149
